In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

catalog = "fortum_challenge_data"
gold_schema = "03_gold"
result_schema = "04_results"
deliverable_schema = "05_deliverables"

historical_input_table = f"{catalog}.{gold_schema}.monthly_training_dataset"
prediction_input_table = f"{catalog}.{result_schema}.consumption_monthly_predictions"

combined_output_table = f"{catalog}.{result_schema}.consumption_monthly_combined"
actuals_output_table = f"{catalog}.{result_schema}.consumption_monthly_actuals"
predictions_output_table = f"{catalog}.{deliverable_schema}.consumption_deliverables_monthly"


def get_sorted_group_ids(df: DataFrame):
    """
    Extract distinct group IDs from a DataFrame and return them in ascending order.

    This list is used to enforce a consistent column order when pivoting the
    data from long format to wide format.

    Args:
        df: Input Spark DataFrame containing a `group_id` column.

    Returns:
        A Python list of distinct group IDs sorted in ascending order.
    """
    return [row["group_id"] for row in df.select("group_id").distinct().orderBy("group_id").collect()]


def aggregate_monthly_consumption(df: DataFrame, source_label: str):
    """
    Aggregate hourly or timestamp-level consumption data into monthly totals.

    The function:
        1. Truncates `timestamp_utc` to the first day of the month
        2. Groups by `group_id` and month
        3. Sums `target_consumption` into `monthly_consumption`
        4. Adds a `source` column to identify whether the data is actual or predicted

    Args:
        df: Input Spark DataFrame containing:
            - `timestamp_utc`
            - `group_id`
            - `target_consumption`
        source_label: Label describing the source of the data, such as
            `"actual"` or `"predicted"`.

    Returns:
        A Spark DataFrame in long format with columns:
            - `group_id`
            - `measured_at`
            - `monthly_consumption`
            - `source`
    """
    return (
        df.withColumn("measured_at", F.date_trunc("month", F.col("timestamp_utc")))
        .groupBy("group_id", "measured_at")
        .agg(F.sum("target_consumption").alias("monthly_consumption"))
        .withColumn("source", F.lit(source_label))
    )


def pivot_monthly_consumption_to_wide(monthly_long_df: DataFrame, group_ids: list[int]):
    """
    Pivot monthly consumption data from long format to wide format.

    The output contains one row per month and one column per `group_id`.
    The column order is determined by the provided `group_ids` list.

    Args:
        monthly_long_df: Spark DataFrame in long format containing:
            - `measured_at`
            - `group_id`
            - `monthly_consumption`
        group_ids: Ordered list of group IDs to use as pivot columns.

    Returns:
        A Spark DataFrame in wide format where:
            - each row represents one month
            - each group_id becomes its own column
            - values are monthly consumption totals
    """
    return (
        monthly_long_df.groupBy("measured_at")
        .pivot("group_id", group_ids)
        .agg(F.first("monthly_consumption"))
        .orderBy("measured_at")
    )


def save_table(df: DataFrame, table_name: str):
    """
    Write a Spark DataFrame to a Delta table using overwrite mode.

    Args:
        df: Spark DataFrame to write.
        table_name: Fully qualified target table name.

    Returns:
        None
    """
    df.write.mode("overwrite").saveAsTable(table_name)


if __name__ == "__main__":


    historical_consumption_df = spark.table(historical_input_table)
    predicted_consumption_df = spark.table(prediction_input_table)

    # Use prediction data to define the output group_id column order
    group_ids = get_sorted_group_ids(predicted_consumption_df)

    # Build monthly actuals table
    monthly_actuals_long_df = aggregate_monthly_consumption(historical_consumption_df, "actual")
    monthly_actuals_wide_df = pivot_monthly_consumption_to_wide(monthly_actuals_long_df, group_ids)

    # Build monthly predictions deliverable
    monthly_predictions_long_df = aggregate_monthly_consumption(predicted_consumption_df, "predicted")
    monthly_predictions_wide_df = pivot_monthly_consumption_to_wide(monthly_predictions_long_df, group_ids)

    # Combine actuals and predictions
    monthly_combined_wide_df = (
        monthly_actuals_wide_df
        .unionByName(monthly_predictions_wide_df)
        .orderBy("measured_at")
    )

    # Save outputs
    save_table(monthly_actuals_wide_df, actuals_output_table)
    save_table(monthly_predictions_wide_df, predictions_output_table)
    save_table(monthly_combined_wide_df, combined_output_table)

    # Preview final result
    display(monthly_combined_wide_df)

